In [ ]:
using Plots
using Printf
using CairoMakie

In [ ]:
data_path = "/home/mfair/JPEC/TODELETE-WandTorqueFilesFromFortran/pentrc_tgar_t_elmat_n1.out"
target_psi = 1.0e-1
psi_tol = 1.0e-10
coeff_name = "A_k" #A, B, C, D, E, and H

In [ ]:
function parse_pentrc_block(path::AbstractString, psi_value::Float64; tol::Float64=1.0e-10)
    in_block = false
    rows = Vector{Tuple{Int,Int,Vector{Float64}}}()
    open(path, "r") do io
        for raw in eachline(io)
            line = strip(raw)
            isempty(line) && continue
            #looks for the psi specified by the user
            if startswith(lowercase(line), "psi") && occursin("=", line)
                psi_str = strip(split(line, "=")[end])
                psi = tryparse(Float64, psi_str)
                psi === nothing && continue
                if in_block
                    break
                end
                in_block = abs(psi - psi_value) <= tol
                continue
            end
            in_block || continue
            if startswith(line, "m_1") || startswith(line, "m_2") || startswith(line, "m_1 m_2")
                continue
            end
            parts = split(line)
            length(parts) < 14 && continue
            m1 = tryparse(Int, parts[1])
            m2 = tryparse(Int, parts[2])
            (m1 === nothing || m2 === nothing) && continue
            vals = Float64[]
            ok = true
            for k in 3:14
                v = tryparse(Float64, parts[k])
                if v === nothing
                    ok = false
                    break
                end
                push!(vals, v)
            end
            ok || continue
            push!(rows, (m1, m2, vals))
        end
    end
    return rows
end

In [ ]:
rows = parse_pentrc_block(data_path, target_psi; tol=psi_tol)
isempty(rows) && error("No data found for psi=$(target_psi)")

coeff_names = ["A_k", "B_k", "C_k", "D_k", "E_k", "H_k"]
coeff_index = findfirst(==(coeff_name), coeff_names)
coeff_index === nothing && error("Unknown coefficient: $(coeff_name)")
col_start = (coeff_index - 1) * 2 + 1
col_end = col_start + 1

m1_vals = sort(unique(first.(rows)))
m2_vals = sort(unique(getindex.(rows, 2)))
m1_index = Dict(m => i for (i, m) in enumerate(m1_vals))
m2_index = Dict(m => i for (i, m) in enumerate(m2_vals))

real_coeff = fill(NaN, length(m1_vals), length(m2_vals))
imag_coeff = fill(NaN, length(m1_vals), length(m2_vals))

for (m1, m2, values) in rows
    i = m1_index[m1]
    j = m2_index[m2]
    real_coeff[i, j] = values[col_start]
    imag_coeff[i, j] = values[col_end]
end

fig = Figure(size = (800, 400))
ax1 = Axis(fig[1, 1], xlabel="m_2", ylabel="m_1", title="real($(coeff_name))")
ax2 = Axis(fig[1, 2], xlabel="m_2", ylabel="m_1", title="imag($(coeff_name))")

p1 = Makie.heatmap!(ax1, m2_vals, m1_vals, real_coeff)
p2 = Makie.heatmap!(ax2, m2_vals, m1_vals, imag_coeff)

Colorbar(fig[1, 3], p1, label = "Value Scale", width = 20)
Label(fig[0, :], "DIII-D", fontsize = 24, font = :bold)

save("kMats_figs/$(coeff_name)_heatmap_DIIID.png", fig)
fig


In [ ]:
# Display real and imaginary parts side by side for clarity
coeffs = real_coeff .+ imag_coeff .* im

fig = Figure(resolution=(1600, 700))

# Real part
ax1 = Axis(fig[1, 1], xlabel="m_2", ylabel="m_1", title="real($(coeff_name))")
hm1 = Makie.heatmap!(ax1, m2_vals, m1_vals, real_coeff; colormap=:RdBu)
Colorbar(fig[1, 2], hm1, label="Real Part", width=20)

# Imaginary part  
ax2 = Axis(fig[1, 3], xlabel="m_2", ylabel="m_1", title="imag($(coeff_name))")
hm2 = Makie.heatmap!(ax2, m2_vals, m1_vals, imag_coeff; colormap=:RdBu)
Colorbar(fig[1, 4], hm2, label="Imaginary Part", width=20)

# Magnitude
ax3 = Axis(fig[1, 5], xlabel="m_2", ylabel="m_1", title="|$(coeff_name)|")
hm3 = Makie.heatmap!(ax3, m2_vals, m1_vals, abs.(coeffs); colormap=:plasma)
Colorbar(fig[1, 6], hm3, label="Magnitude", width=20)

fig


In [ ]:
coeffs = real_coeff .+ imag_coeff .* im
coeff_mag = abs.(coeffs)

fig = Makie.Figure(resolution=(1400, 900))
ax = Makie.Axis(fig[1, 1]; title="magnitude($(coeff_name))", xlabel="m_2", ylabel="m_1")

hm = Makie.heatmap!(ax, m2_vals, m1_vals, coeff_mag; colormap=:plasma)
Makie.Colorbar(fig[1, 2], hm; label="|$(coeff_name)|")

positions = [Point2f(m2_vals[j], m1_vals[i]) for i in eachindex(m1_vals), j in eachindex(m2_vals)]
directions = [Point2f(real_coeff[i, j], imag_coeff[i, j]) for i in eachindex(m1_vals), j in eachindex(m2_vals)]
Makie.arrows2d!(ax, vec(positions), vec(directions); lengthscale=0.6, tipwidth=0.1, tiplength=0.1, color=:white)

fig

In [ ]:
coeffs = real_coeff .+ imag_coeff .* im
coeff_phase = angle.(coeffs)
coeff_mag = abs.(coeffs)

fig = Makie.Figure(resolution=(1400, 900))
ax = Makie.Axis(fig[1, 1]; title="phase($(coeff_name))", xlabel="m_2", ylabel="m_1")

hm = Makie.heatmap!(ax, m2_vals, m1_vals, coeff_mag; colormap=:plasma)
Makie.Colorbar(fig[1, 2], hm; label="angle($(coeff_name))")

positions = [Point2f(m2_vals[j], m1_vals[i]) for i in eachindex(m1_vals), j in eachindex(m2_vals)]
directions = [Point2f(real_coeff[i, j], imag_coeff[i, j]) for i in eachindex(m1_vals), j in eachindex(m2_vals)]
Makie.arrows2d!(ax, vec(positions), vec(directions); lengthscale=0.6, tipwidth=0.04, tiplength=0.06, color=:white)

fig